In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import os

BASE_DIR = r"C:\Users\Hunter\OneDrive - Grand Canyon University\Documents\GitHub Projects\gcu-capstone-project"
DATA_DIR = os.path.join(BASE_DIR, "data")
OUT_DIR  = os.path.join(BASE_DIR, "outputs")

TARGET_RED  = "#CC0000"
TARGET_GRAY = "#666666"

print("✅ Imports OK")

✅ Imports OK


In [2]:
orders    = pd.read_csv(os.path.join(DATA_DIR, "olist_orders_dataset.csv"), parse_dates=["order_purchase_timestamp"])
customers = pd.read_csv(os.path.join(DATA_DIR, "olist_customers_dataset.csv"))
payments  = pd.read_csv(os.path.join(DATA_DIR, "olist_order_payments_dataset.csv"))
items     = pd.read_csv(os.path.join(DATA_DIR, "olist_order_items_dataset.csv"))
reviews   = pd.read_csv(os.path.join(DATA_DIR, "olist_order_reviews_dataset.csv"))

print("✅ All tables loaded")

✅ All tables loaded


In [3]:
# Delivered orders only
orders_delivered = orders[orders["order_status"] == "delivered"].copy()

# Monetary: total payment per order
order_value = payments.groupby("order_id")["payment_value"].sum().reset_index()
order_value.columns = ["order_id", "order_total"]

# Payment type: dominant payment method per order
order_payment_type = payments.groupby("order_id")["payment_type"].agg(
    lambda x: x.value_counts().index[0]
).reset_index()
order_payment_type.columns = ["order_id", "payment_type"]

# Freight: avg freight value per order
order_freight = items.groupby("order_id")["freight_value"].sum().reset_index()
order_freight.columns = ["order_id", "freight_value"]

# Review score: one review per order (take max if duplicates)
order_review = reviews.groupby("order_id")["review_score"].max().reset_index()

# Merge everything onto delivered orders
df = orders_delivered.merge(customers[["customer_id", "customer_unique_id"]], on="customer_id")
df = df.merge(order_value, on="order_id")
df = df.merge(order_payment_type, on="order_id", how="left")
df = df.merge(order_freight, on="order_id", how="left")
df = df.merge(order_review, on="order_id", how="left")

print(f"Merged shape: {df.shape}")
print(f"Null check:\n{df[['order_total','payment_type','freight_value','review_score']].isnull().sum()}")
print("✅ Merge complete")

Merged shape: (96477, 13)
Null check:
order_total        0
payment_type       0
freight_value      0
review_score     646
dtype: int64
✅ Merge complete


In [4]:
snapshot_date = df["order_purchase_timestamp"].max() + pd.Timedelta(days=1)

rfm_plus = df.groupby("customer_unique_id").agg(
    Recency       =("order_purchase_timestamp", lambda x: (snapshot_date - x.max()).days),
    Frequency     =("order_id", "count"),
    Monetary      =("order_total", "sum"),
    Avg_Review    =("review_score", "mean"),
    Avg_Freight   =("freight_value", "mean"),
    Payment_Type  =("payment_type", lambda x: x.value_counts().index[0]),
    Tenure        =("order_purchase_timestamp", lambda x: (x.max() - x.min()).days)
).reset_index()

print(f"RFM+ table shape: {rfm_plus.shape}")
print(rfm_plus.describe(include="all").round(2))
print("✅ RFM+ table built")

RFM+ table shape: (93357, 8)
                      customer_unique_id   Recency  Frequency  Monetary  \
count                              93357  93357.00   93357.00  93357.00   
unique                             93357       NaN        NaN       NaN   
top     0000366f3b9a7992bf8c76cfdf3221e2       NaN        NaN       NaN   
freq                                   1       NaN        NaN       NaN   
mean                                 NaN    237.94       1.03    165.20   
std                                  NaN    152.58       0.21    226.31   
min                                  NaN      1.00       1.00      9.59   
25%                                  NaN    114.00       1.00     63.06   
50%                                  NaN    219.00       1.00    107.78   
75%                                  NaN    346.00       1.00    182.56   
max                                  NaN    695.00      15.00  13664.08   

        Avg_Review  Avg_Freight Payment_Type    Tenure  
count     927

In [5]:
# Handle missing review scores
# 603 customers have no review — impute with dataset median
median_review = rfm_plus["Avg_Review"].median()
rfm_plus["Avg_Review"] = rfm_plus["Avg_Review"].fillna(median_review)

print(f"Imputed {rfm_plus['Avg_Review'].isnull().sum()} nulls with median review score: {median_review}")
print(f"Null check after imputation:\n{rfm_plus.isnull().sum()}")
print("✅ Nulls resolved")

Imputed 0 nulls with median review score: 5.0
Null check after imputation:
customer_unique_id    0
Recency               0
Frequency             0
Monetary              0
Avg_Review            0
Avg_Freight           0
Payment_Type          0
Tenure                0
dtype: int64
✅ Nulls resolved


In [6]:
# Simpson's Rule Check
# Does avg review score differ by payment type subgroup?
simpson_check = rfm_plus.groupby("Payment_Type").agg(
    Customer_Count=("Avg_Review", "count"),
    Avg_Review_Score=("Avg_Review", "mean"),
    Avg_Monetary=("Monetary", "mean"),
    Avg_Recency=("Recency", "mean")
).round(2).reset_index()

print("Simpson's Rule Check — RFM Behavior by Payment Type:")
print(simpson_check.to_string(index=False))
print("\n✅ Simpson Rule check complete")

Simpson's Rule Check — RFM Behavior by Payment Type:
Payment_Type  Customer_Count  Avg_Review_Score  Avg_Monetary  Avg_Recency
      boleto           18592              4.16        148.99       247.50
 credit_card           70756              4.16        171.12       236.11
  debit_card            1444              4.24        144.32       168.14
     voucher            2565              4.13        131.22       258.28

✅ Simpson Rule check complete


In [7]:
from sklearn.preprocessing import StandardScaler, LabelEncoder

# Encode Payment_Type
le = LabelEncoder()
rfm_plus["Payment_Type_Encoded"] = le.fit_transform(rfm_plus["Payment_Type"])
print(f"Payment type encoding: {dict(zip(le.classes_, le.transform(le.classes_)))}")

# Select features for clustering
features = ["Recency", "Frequency", "Monetary", "Avg_Review", "Avg_Freight", "Payment_Type_Encoded", "Tenure"]
X = rfm_plus[features].copy()

# Scale
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f"\nScaled feature matrix shape: {X_scaled.shape}")
print(f"Mean after scaling (should be ~0): {X_scaled.mean(axis=0).round(3)}")
print(f"Std after scaling (should be ~1):  {X_scaled.std(axis=0).round(3)}")
print("✅ Scaling complete")

Payment type encoding: {'boleto': np.int64(0), 'credit_card': np.int64(1), 'debit_card': np.int64(2), 'voucher': np.int64(3)}

Scaled feature matrix shape: (93357, 7)
Mean after scaling (should be ~0): [ 0. -0. -0.  0.  0. -0. -0.]
Std after scaling (should be ~1):  [1. 1. 1. 1. 1. 1. 1.]
✅ Scaling complete


In [8]:
# Export cleaned RFM+ table
rfm_plus_export = rfm_plus.copy()

export_path = os.path.join(OUT_DIR, "rfm_plus_table.csv")
rfm_plus_export.to_csv(export_path, index=False)

print(f"✅ RFM+ table exported: {len(rfm_plus_export):,} rows")
print(f"   Columns: {list(rfm_plus_export.columns)}")
print(f"   Saved to: {export_path}")

✅ RFM+ table exported: 93,357 rows
   Columns: ['customer_unique_id', 'Recency', 'Frequency', 'Monetary', 'Avg_Review', 'Avg_Freight', 'Payment_Type', 'Tenure', 'Payment_Type_Encoded']
   Saved to: C:\Users\Hunter\OneDrive - Grand Canyon University\Documents\GitHub Projects\gcu-capstone-project\outputs\rfm_plus_table.csv
